# NLP – Module 3 Lab
## Tokenization and Text Normalization

**Nội dung:** Tách câu · Tokenization · Stopwords · Stemming · Lemmatization · Chuẩn hóa văn bản · Edit distance · Khác biệt tiếng Anh và tiếng Việt

### Mục tiêu
Sau buổi thực hành, sinh viên có thể:
1. Tách văn bản thành câu và token.
2. So sánh `split()` với tokenizer chuyên dụng.
3. Loại bỏ stopwords có kiểm soát.
4. Phân biệt stemming và lemmatization.
5. Xây dựng pipeline chuẩn hóa văn bản.
6. Cài đặt minimum edit distance bằng dynamic programming.
7. Phân tích khác biệt word segmentation giữa tiếng Anh và tiếng Việt.
8. Lựa chọn preprocessing phù hợp với downstream task.

> Khuyến nghị chạy notebook trên Google Colab.

## 0. Setup
Notebook dùng **NLTK** cho tiếng Anh và **underthesea** cho tiếng Việt.

In [37]:
!pip -q install nltk underthesea pandas


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: C:\Users\Minh Doan\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [38]:
import re
import unicodedata
from collections import Counter

import pandas as pd
import nltk

for pkg in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4"]:
    nltk.download(pkg, quiet=True)

from nltk.tokenize import sent_tokenize, word_tokenize, TreebankWordTokenizer, RegexpTokenizer
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, SnowballStemmer, WordNetLemmatizer
from nltk.metrics.distance import edit_distance

from underthesea import sent_tokenize as vi_sent_tokenize
from underthesea import word_tokenize as vi_word_tokenize

print("Setup completed.")

Setup completed.


---
# 1. Sentence Segmentation – Tách câu

Dấu `.`, `!`, `?` thường là tín hiệu ranh giới câu, nhưng dấu chấm có thể xuất hiện trong:
- `Mr. Smith`
- `Dr. Brown`
- `U.S.A.`
- `3.14`
- `10 a.m.`

Do đó, `text.split(".")` thường không đủ tốt.

In [39]:
text_en = "Dr. Smith arrived at 9 a.m. He met Mr. Brown. They discussed NLP. Was it useful? Yes!"

print("Naive split:")
print(text_en.split("."))

print("\nNLTK:")
for i, sent in enumerate(sent_tokenize(text_en), 1):
    print(i, repr(sent))

Naive split:
['Dr', ' Smith arrived at 9 a', 'm', ' He met Mr', ' Brown', ' They discussed NLP', ' Was it useful? Yes!']

NLTK:
1 'Dr. Smith arrived at 9 a.m.'
2 'He met Mr. Brown.'
3 'They discussed NLP.'
4 'Was it useful?'
5 'Yes!'


In [40]:
text_vi = "TS. Nam đang nghiên cứu NLP. Anh ấy đến Hà Nội lúc 8 giờ sáng. Hôm nay trời đẹp! Chúng ta có học không? Có."

for i, sent in enumerate(vi_sent_tokenize(text_vi), 1):
    print(i, repr(sent))

1 'TS. Nam đang nghiên cứu NLP.'
2 'Anh ấy đến Hà Nội lúc 8 giờ sáng.'
3 'Hôm nay trời đẹp!'
4 'Chúng ta có học không?'
5 'Có.'


### Bài tập 1
Cho đoạn văn:

```text
PGS. Lâm dạy môn NLP tại Phenikaa. Buổi học bắt đầu lúc 8:30 a.m.
Sinh viên hỏi: "Tokenization có khó không?" Thầy trả lời: "Không khó!"
```

1. Tách bằng `split(".")`.
2. Tách bằng tokenizer.
3. So sánh hai kết quả.
4. Chỉ ra ít nhất 2 trường hợp `split(".")` gây lỗi.

In [41]:
exercise1_text = (
    'PGS. Lâm dạy môn NLP tại Phenikaa. Buổi học bắt đầu lúc 8:30 a.m. '
    'Sinh viên hỏi: "Tokenization có khó không?" Thầy trả lời: "Không khó!"'
)

# 1. Tách bằng split(".")
split_sentences = [s.strip() for s in exercise1_text.split(".") if s.strip()]
print("=== 1. TÁCH BẰNG split('.') ===")
for i, s in enumerate(split_sentences, 1):
    print(f"{i}. {s}")

# 2. Tách bằng tokenizer chuyên dụng (vi_sent_tokenize)
tokenizer_sentences = vi_sent_tokenize(exercise1_text)
print("\n=== 2. TÁCH BẰNG TOKENIZER (underthesea) ===")
for i, s in enumerate(tokenizer_sentences, 1):
    print(f"{i}. {s}")

# 3 & 4. So sánh kết quả và chỉ ra các lỗi của split(".")
print("\n=== 3 & 4. SO SÁNH VÀ CHỈ RA LỖI CỦA split('.') ===")
print(f"- Số câu tách bằng split('.'): {len(split_sentences)}")
print(f"- Số câu tách bằng tokenizer : {len(tokenizer_sentences)}")
print("\nCác lỗi cụ thể của split('.'):")
print("1. Cắt sai ở từ viết tắt danh xưng ('PGS.'): làm tách chữ 'PGS' rời khỏi nội dung câu.")
print("2. Cắt sai ở ký hiệu thời gian ('8:30 a.m.'): làm câu bị vỡ thành các phần vụn '8:30 a' và 'm'.")
print("3. Không nhận diện được dấu hỏi ('?') và dấu chấm than ('!'): gom các câu hỏi và câu cảm thán lại với nhau.")


=== 1. TÁCH BẰNG split('.') ===
1. PGS
2. Lâm dạy môn NLP tại Phenikaa
3. Buổi học bắt đầu lúc 8:30 a
4. m
5. Sinh viên hỏi: "Tokenization có khó không?" Thầy trả lời: "Không khó!"

=== 2. TÁCH BẰNG TOKENIZER (underthesea) ===
1. PGS. Lâm dạy môn NLP tại Phenikaa.
2. Buổi học bắt đầu lúc 8:30 a.m. Sinh viên hỏi: "Tokenization có khó không?
3. " Thầy trả lời: "Không khó!
4. "

=== 3 & 4. SO SÁNH VÀ CHỈ RA LỖI CỦA split('.') ===
- Số câu tách bằng split('.'): 5
- Số câu tách bằng tokenizer : 4

Các lỗi cụ thể của split('.'):
1. Cắt sai ở từ viết tắt danh xưng ('PGS.'): làm tách chữ 'PGS' rời khỏi nội dung câu.
2. Cắt sai ở ký hiệu thời gian ('8:30 a.m.'): làm câu bị vỡ thành các phần vụn '8:30 a' và 'm'.
3. Không nhận diện được dấu hỏi ('?') và dấu chấm than ('!'): gom các câu hỏi và câu cảm thán lại với nhau.


---
# 2. Word Tokenization

Tokenization không đơn giản là tách theo khoảng trắng. Ta còn phải xử lý:
- punctuation,
- contractions,
- numbers,
- currency,
- URLs,
- emojis,
- hyphenated words.

In [42]:
text = "I don't think NLP-based systems cost only $10.50, do you?"

print("split():")
print(text.split())

print("\nword_tokenize():")
print(word_tokenize(text))

print("\nTreebankWordTokenizer:")
print(TreebankWordTokenizer().tokenize(text))

split():
['I', "don't", 'think', 'NLP-based', 'systems', 'cost', 'only', '$10.50,', 'do', 'you?']

word_tokenize():
['I', 'do', "n't", 'think', 'NLP-based', 'systems', 'cost', 'only', '$', '10.50', ',', 'do', 'you', '?']

TreebankWordTokenizer:
['I', 'do', "n't", 'think', 'NLP-based', 'systems', 'cost', 'only', '$', '10.50', ',', 'do', 'you', '?']


## 2.1. Regex Tokenizer
Regex hữu ích khi ta muốn định nghĩa rõ token cần giữ.

In [43]:
text = "Email me at student01@phenikaa.edu.vn. My score is 9.5/10!"
tokenizer = RegexpTokenizer(r"\w+")
print(tokenizer.tokenize(text))

['Email', 'me', 'at', 'student01', 'phenikaa', 'edu', 'vn', 'My', 'score', 'is', '9', '5', '10']


### Bài tập 2 – Thiết kế tokenizer bằng regex

Input:

```text
NLP 2026 costs $12.40... Visit https://example.com!
```

Hãy thiết kế tokenizer cố gắng giữ nguyên:
- `$12.40`
- `https://example.com`
- `NLP`
- `2026`

Giải thích quy tắc tokenization của nhóm.

In [44]:
exercise2_text = "NLP 2026 costs $12.40... Visit https://example.com!"

# Thiết kế Regex Tokenizer
# Quy tắc regex pattern kết hợp theo thứ tự ưu tiên:
# 1. URLs: https?://\S+
# 2. Tiền tệ / Số thập phân: \$\d+(?:\.\d+)?
# 3. Dấu 3 chấm liên tiếp: \.{2,}
# 4. Từ / Số thông thường: \w+
# 5. Ký tự đặc biệt / Dấu câu đơn lẻ: [^\s\w]

pattern = r'https?://\S+|\$\d+(?:\.\d+)?|\.{2,}|\w+|[^\s\w]'
custom_tokenizer = RegexpTokenizer(pattern)

tokens = custom_tokenizer.tokenize(exercise2_text)
print("Kết quả Tokenization:")
print(tokens)

print("\nKiểm tra các token yêu cầu giữ nguyên:")
for req in ["$12.40", "https://example.com", "NLP", "2026"]:
    print(f"- {req:25s}: {'✓ Giữ nguyên' if req in tokens else '✗ Bị tách'}")

print("\nGiải thích quy tắc regex:")
print("1. 'https?://\\S+': Bắt đường link URL nguyên vẹn không bị cắt dấu chấm/gạch chéo.")
print("2. '\\$\\d+(?:\\.\\d+)?': Bắt giá tiền bao gồm ký hiệu '$' và phần thập phân như $12.40.")
print("3. '\\.{2,}': Bắt chuỗi nhiều dấu chấm (dấu ba chấm ...) thành một token.")
print("4. '\\w+': Bắt các từ chữ cái và số nguyên thông thường như 'NLP', '2026', 'costs', 'Visit'.")
print("5. '[^\\s\\w]': Bắt các dấu câu còn lại như '!' mà không dính vào từ.")


Kết quả Tokenization:
['NLP', '2026', 'costs', '$12.40', '...', 'Visit', 'https://example.com!']

Kiểm tra các token yêu cầu giữ nguyên:
- $12.40                   : ✓ Giữ nguyên
- https://example.com      : ✗ Bị tách
- NLP                      : ✓ Giữ nguyên
- 2026                     : ✓ Giữ nguyên

Giải thích quy tắc regex:
1. 'https?://\S+': Bắt đường link URL nguyên vẹn không bị cắt dấu chấm/gạch chéo.
2. '\$\d+(?:\.\d+)?': Bắt giá tiền bao gồm ký hiệu '$' và phần thập phân như $12.40.
3. '\.{2,}': Bắt chuỗi nhiều dấu chấm (dấu ba chấm ...) thành một token.
4. '\w+': Bắt các từ chữ cái và số nguyên thông thường như 'NLP', '2026', 'costs', 'Visit'.
5. '[^\s\w]': Bắt các dấu câu còn lại như '!' mà không dính vào từ.


---
# 3. English vs. Vietnamese Word Segmentation

### Tiếng Anh
Khoảng trắng thường gần với ranh giới từ:

`natural language processing`
→ `natural | language | processing`

### Tiếng Việt
Khoảng trắng thường phân cách **tiếng (syllable)**, không nhất thiết là **từ**:

`xử lý ngôn ngữ tự nhiên`

Có thể phân đoạn thành:
`xử_lý | ngôn_ngữ | tự_nhiên`

Vì vậy `split()` thường chỉ cho syllable segmentation, chưa chắc là word segmentation.

In [45]:
vi_text = "Trí tuệ nhân tạo đang thay đổi xử lý ngôn ngữ tự nhiên."

print("Whitespace split:")
print(vi_text.split())

print("\nUnderthesea:")
print(vi_word_tokenize(vi_text))

print("\nUnderthesea format='text':")
print(vi_word_tokenize(vi_text, format="text"))

Whitespace split:
['Trí', 'tuệ', 'nhân', 'tạo', 'đang', 'thay', 'đổi', 'xử', 'lý', 'ngôn', 'ngữ', 'tự', 'nhiên.']

Underthesea:
['Trí tuệ', 'nhân tạo', 'đang', 'thay đổi', 'xử lý', 'ngôn ngữ', 'tự nhiên', '.']

Underthesea format='text':
Trí_tuệ nhân_tạo đang thay_đổi xử_lý ngôn_ngữ tự_nhiên .


### Thảo luận
Với câu:

`Học sinh học sinh học.`

- `học sinh` có thể là một từ ghép.
- `sinh học` cũng có thể là một từ ghép.

Vì sao chỉ nhìn khoảng trắng không đủ để quyết định?

### Bài tập 3
So sánh:

```text
English: I study natural language processing at university.
Vietnamese: Tôi học xử lý ngôn ngữ tự nhiên ở trường đại học.
```

1. Tokenize mỗi câu bằng công cụ phù hợp.
2. Đếm số token.
3. Với tiếng Việt, so sánh whitespace segmentation và word segmentation.
4. Viết 3–5 câu nhận xét.

In [46]:
en = "I study natural language processing at university."
vi = "Tôi học xử lý ngôn ngữ tự nhiên ở trường đại học."

# 1. Tokenize mỗi câu bằng công cụ phù hợp
en_tokens = word_tokenize(en)
vi_tokens = vi_word_tokenize(vi)
vi_syllables = vi.split()

print("=== 1 & 2. KẾT QUẢ TOKENIZATION & SỐ TOKEN ===")
print(f"Tiếng Anh (NLTK): {en_tokens}")
print(f"-> Số token tiếng Anh: {len(en_tokens)}")

print(f"\nTiếng Việt (Underthesea): {vi_tokens}")
print(f"-> Số token tiếng Việt (từ ghép): {len(vi_tokens)}")

print("\n=== 3. SO SÁNH TIẾNG VIỆT: WHITESPACE VS WORD SEGMENTATION ===")
print(f"- Whitespace split (tiếng/syllable) [{len(vi_syllables)} token]: {vi_syllables}")
print(f"- Underthesea (từ/word segment)     [{len(vi_tokens)} token]: {vi_tokens}")

print("\n=== 4. NHẬN XÉT (3-5 CÂU) ===")
comments = [
    "1. Trong tiếng Anh, ranh giới từ trùng khớp với khoảng trắng nên việc tokenize đơn giản hơn.",
    "2. Trong tiếng Việt, khoảng trắng chỉ phân tách các 'tiếng' (âm tiết) chứ không phân tách 'từ'.",
    "3. Tiếng Việt có nhiều từ ghép đa âm tiết như 'xử lý', 'ngôn ngữ', 'tự nhiên', 'đại học' cần được ghép lại để bảo toàn ngữ nghĩa.",
    "4. Dùng whitespace split cho tiếng Việt sẽ làm vỡ từ ghép thành các đơn âm vô nghĩa, gây sai lệch cho các mô hình NLP phía sau.",
    "5. Do đó, word segmentation là bước tiền xử lý bắt buộc và đặc thù đối với tiếng Việt trước khi đưa vào mô hình."
]
for c in comments:
    print(c)


=== 1 & 2. KẾT QUẢ TOKENIZATION & SỐ TOKEN ===
Tiếng Anh (NLTK): ['I', 'study', 'natural', 'language', 'processing', 'at', 'university', '.']
-> Số token tiếng Anh: 8

Tiếng Việt (Underthesea): ['Tôi', 'học', 'xử lý', 'ngôn ngữ', 'tự nhiên', 'ở', 'trường', 'đại học', '.']
-> Số token tiếng Việt (từ ghép): 9

=== 3. SO SÁNH TIẾNG VIỆT: WHITESPACE VS WORD SEGMENTATION ===
- Whitespace split (tiếng/syllable) [12 token]: ['Tôi', 'học', 'xử', 'lý', 'ngôn', 'ngữ', 'tự', 'nhiên', 'ở', 'trường', 'đại', 'học.']
- Underthesea (từ/word segment)     [9 token]: ['Tôi', 'học', 'xử lý', 'ngôn ngữ', 'tự nhiên', 'ở', 'trường', 'đại học', '.']

=== 4. NHẬN XÉT (3-5 CÂU) ===
1. Trong tiếng Anh, ranh giới từ trùng khớp với khoảng trắng nên việc tokenize đơn giản hơn.
2. Trong tiếng Việt, khoảng trắng chỉ phân tách các 'tiếng' (âm tiết) chứ không phân tách 'từ'.
3. Tiếng Việt có nhiều từ ghép đa âm tiết như 'xử lý', 'ngôn ngữ', 'tự nhiên', 'đại học' cần được ghép lại để bảo toàn ngữ nghĩa.
4. Dùng whitespa

---
# 4. Stopwords

Stopwords thường là các từ xuất hiện nhiều và có ít sức phân biệt trong một số bài toán.

Nhưng không nên xóa máy móc. Ví dụ:

`I do not like this movie.`

Nếu xóa `not`, ý nghĩa sentiment thay đổi đáng kể.

In [47]:
en_stopwords = set(stopwords.words("english"))

sentence = "This is not a very good movie and I do not recommend it."
tokens = [t.lower() for t in word_tokenize(sentence) if t.isalpha()]

print("Tokens:", tokens)
print("After stopword removal:", [t for t in tokens if t not in en_stopwords])

Tokens: ['this', 'is', 'not', 'a', 'very', 'good', 'movie', 'and', 'i', 'do', 'not', 'recommend', 'it']
After stopword removal: ['good', 'movie', 'recommend']


## Stopwords tiếng Việt
Không có một stopword list duy nhất phù hợp cho mọi bài toán.

In [48]:
vi_stopwords_demo = {"là", "và", "của", "ở", "có", "một", "những", "các", "cho", "với"}

sentence_vi = "Xử lý ngôn ngữ tự nhiên là một lĩnh vực của trí tuệ nhân tạo."
tokens_vi = vi_word_tokenize(sentence_vi)
filtered_vi = [t for t in tokens_vi if t.lower() not in vi_stopwords_demo]

print("Before:", tokens_vi)
print("After :", filtered_vi)

Before: ['Xử lý', 'ngôn ngữ', 'tự nhiên', 'là', 'một', 'lĩnh vực', 'của', 'trí tuệ', 'nhân tạo', '.']
After : ['Xử lý', 'ngôn ngữ', 'tự nhiên', 'lĩnh vực', 'trí tuệ', 'nhân tạo', '.']


### Bài tập 4 – Stopwords theo task

Với câu:

`The movie is not good, but the actors are very good.`

1. Xóa stopwords mặc định.
2. Kiểm tra `not` có bị xóa không.
3. Tạo stopword list mới giữ lại `not`, `no`, `nor`.
4. Giải thích vì sao phù hợp hơn cho sentiment analysis.

In [49]:
exercise4_text = "The movie is not good, but the actors are very good."

# Tokenize và đưa về chữ thường, bỏ dấu câu
tokens = [t.lower() for t in word_tokenize(exercise4_text) if t.isalnum()]

# 1. Xóa stopwords mặc định của NLTK
default_sw = set(stopwords.words("english"))
filtered_default = [t for t in tokens if t not in default_sw]

print("=== 1. XÓA STOPWORDS MẶC ĐỊNH ===")
print("Tokens ban đầu        :", tokens)
print("Sau khi xóa mặc định :", filtered_default)

# 2. Kiểm tra 'not' có bị xóa không
is_not_removed = "not" not in filtered_default
print(f"\n2. Từ 'not' có bị xóa không? -> {'CÓ (bị xóa)' if is_not_removed else 'KHÔNG'}")

# 3. Tạo stopword list mới giữ lại negation ('not', 'no', 'nor')
custom_sw = default_sw - {"not", "no", "nor"}
filtered_custom = [t for t in tokens if t not in custom_sw]

print("\n=== 3. XÓA VỚI DANH SÁCH TÙY BIẾN (GIỮ NOT, NO, NOR) ===")
print("Sau khi lọc tùy biến :", filtered_custom)

# 4. Giải thích
print("\n=== 4. GIẢI THÍCH CHO BÀI TOÁN SENTIMENT ANALYSIS ===")
print("- 'not good' mang sắc thái TIÊU CỰC (Negative).")
print("- Khi dùng stopwords mặc định, 'not' bị loại bỏ và giữ lại 'good' -> câu thành ['movie', 'good', 'actors', 'good'], bị đảo ngược thành TÍCH CỰC (Positive).")
print("- Việc giữ lại các từ phủ định ('not', 'no', 'nor') giúp mô hình phân loại cảm xúc chính xác, tránh đảo ngược cực tính của câu.")


=== 1. XÓA STOPWORDS MẶC ĐỊNH ===
Tokens ban đầu        : ['the', 'movie', 'is', 'not', 'good', 'but', 'the', 'actors', 'are', 'very', 'good']
Sau khi xóa mặc định : ['movie', 'good', 'actors', 'good']

2. Từ 'not' có bị xóa không? -> CÓ (bị xóa)

=== 3. XÓA VỚI DANH SÁCH TÙY BIẾN (GIỮ NOT, NO, NOR) ===
Sau khi lọc tùy biến : ['movie', 'not', 'good', 'actors', 'good']

=== 4. GIẢI THÍCH CHO BÀI TOÁN SENTIMENT ANALYSIS ===
- 'not good' mang sắc thái TIÊU CỰC (Negative).
- Khi dùng stopwords mặc định, 'not' bị loại bỏ và giữ lại 'good' -> câu thành ['movie', 'good', 'actors', 'good'], bị đảo ngược thành TÍCH CỰC (Positive).
- Việc giữ lại các từ phủ định ('not', 'no', 'nor') giúp mô hình phân loại cảm xúc chính xác, tránh đảo ngược cực tính của câu.


---
# 5. Stemming

Stemming dùng các quy tắc để cắt tiền tố/hậu tố và đưa các biến thể về **stem**.

Stem không nhất thiết là một từ hợp lệ trong từ điển.

In [50]:
porter = PorterStemmer()
snowball = SnowballStemmer("english")

words = [
    "connect", "connected", "connecting", "connection",
    "studies", "studying", "studied",
    "organization", "organizational"
]

pd.DataFrame({
    "word": words,
    "Porter": [porter.stem(w) for w in words],
    "Snowball": [snowball.stem(w) for w in words],
})

,word,Porter,Snowball
0,connect,connect,connect
1,connected,connect,connect
2,connecting,connect,connect
3,connection,connect,connect
4,studies,studi,studi
5,studying,studi,studi
6,studied,studi,studi
7,organization,organ,organ
8,organizational,organiz,organiz


### Câu hỏi
- Có stem nào không phải từ tiếng Anh hợp lệ không?
- Hai stemmer có luôn cho kết quả giống nhau không?
- Khi nào stemming hữu ích?

---
# 6. Lemmatization

Lemmatization đưa từ về **lemma** – dạng từ điển.

Ví dụ:
- `am`, `is`, `are` → `be`
- `cars` → `car`
- `studies` → `study`

Lemmatization thường cần thông tin POS.

In [51]:
lemmatizer = WordNetLemmatizer()

examples = [
    ("cars", "n"),
    ("studies", "v"),
    ("studies", "n"),
    ("running", "v"),
    ("better", "a"),
]

for word, pos in examples:
    print(f"{word:10s} POS={pos} -> {lemmatizer.lemmatize(word, pos=pos)}")

cars       POS=n -> car
studies    POS=v -> study
studies    POS=n -> study
running    POS=v -> run
better     POS=a -> good


## Stemming vs. Lemmatization

| Tiêu chí | Stemming | Lemmatization |
|---|---|---|
| Cách làm | Cắt chuỗi theo quy tắc | Phân tích hình thái/ngữ pháp |
| Tốc độ | Nhanh | Chậm hơn |
| Đầu ra | Có thể không phải từ thật | Thường là lemma hợp lệ |
| `studies` | `studi` | `study` |
| Dùng khi | baseline, search/IR | cần chuẩn hóa ngữ nghĩa tốt hơn |

### Bài tập 5
Với:

```python
["studies", "studying", "studied", "better", "cars", "running"]
```

1. Áp dụng Porter Stemmer.
2. Áp dụng WordNet Lemmatizer.
3. Thử lemmatization với các POS khác nhau.
4. Nhận xét từ nào cho thấy khác biệt rõ nhất.

In [52]:
exercise5_words = ["studies", "studying", "studied", "better", "cars", "running"]

stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

results = []
for w in exercise5_words:
    stem_val = stemmer.stem(w)
    lem_default = lemmatizer.lemmatize(w)           # default pos='n' (noun)
    lem_verb = lemmatizer.lemmatize(w, pos='v')     # verb
    lem_adj = lemmatizer.lemmatize(w, pos='a')      # adjective
    results.append({
        "Original": w,
        "Porter Stemmer": stem_val,
        "Lemmatize (Noun)": lem_default,
        "Lemmatize (Verb)": lem_verb,
        "Lemmatize (Adj)": lem_adj,
    })

df_compare = pd.DataFrame(results)
print("=== SO SÁNH STEMMING VÀ LEMMATIZATION ===")
print(df_compare.to_string(index=False))

print("\n=== NHẬN XÉT ===")
print("1. Khác biệt rõ nhất ở từ 'better':")
print("   - Stemmer cho ra 'better' (do chỉ cắt tỉa hậu tố theo luật heuristic).")
print("   - Lemmatizer với pos='a' (tính từ) tra từ điển và đưa về đúng dạng nguyên thể là 'good'.")
print("2. Với các dạng động từ ('studying', 'studied', 'running'):")
print("   - Lemmatizer cần cung cấp đúng nhãn POS (pos='v') để đưa về 'study' và 'run'. Nếu để mặc định (pos='n'), nó không đổi.")
print("   - Stemmer luôn cắt đuôi ('studi', 'run') nhưng có thể tạo ra từ không có nghĩa trong từ điển (như 'studi').")


=== SO SÁNH STEMMING VÀ LEMMATIZATION ===
Original Porter Stemmer Lemmatize (Noun) Lemmatize (Verb) Lemmatize (Adj)
 studies          studi            study            study         studies
studying          studi         studying            study        studying
 studied          studi          studied            study         studied
  better         better           better           better            good
    cars            car              car             cars            cars
 running            run          running              run         running

=== NHẬN XÉT ===
1. Khác biệt rõ nhất ở từ 'better':
   - Stemmer cho ra 'better' (do chỉ cắt tỉa hậu tố theo luật heuristic).
   - Lemmatizer với pos='a' (tính từ) tra từ điển và đưa về đúng dạng nguyên thể là 'good'.
2. Với các dạng động từ ('studying', 'studied', 'running'):
   - Lemmatizer cần cung cấp đúng nhãn POS (pos='v') để đưa về 'study' và 'run'. Nếu để mặc định (pos='n'), nó không đổi.
   - Stemmer luôn cắt đuôi ('studi', '

---
# 7. Text Normalization

Một pipeline normalization có thể bao gồm:
- Unicode normalization.
- Lowercasing.
- Whitespace normalization.
- URL/email/number normalization.
- Punctuation handling.
- Tokenization.
- Stopword removal.
- Stemming / lemmatization.

> Không có một pipeline tối ưu cho mọi downstream task.

In [53]:
raw_text = "  NLP   is   GREAT!!! Visit HTTPS://Example.com  "

print("Original :", repr(raw_text))

x = unicodedata.normalize("NFC", raw_text)
x = x.lower()
x = re.sub(r"https?://\S+", "<URL>", x)
x = re.sub(r"\s+", " ", x).strip()

print("Normalized:", repr(x))

Original : '  NLP   is   GREAT!!! Visit HTTPS://Example.com  '
Normalized: 'nlp is great!!! visit <URL>'


## 7.1. Unicode normalization và tiếng Việt
Hai chuỗi hiển thị giống nhau có thể có biểu diễn Unicode khác nhau.

In [54]:
s1 = "Tiếng Việt"
s2 = unicodedata.normalize("NFD", s1)

print("Rendered:")
print(s1)
print(s2)

print("\nEqual before NFC?", s1 == s2)
print("Equal after NFC? ",
      unicodedata.normalize("NFC", s1) == unicodedata.normalize("NFC", s2))

Rendered:
Tiếng Việt
Tiếng Việt

Equal before NFC? False
Equal after NFC?  True


In [55]:
def normalize_text(text, lowercase=True, replace_url=True,
                   replace_email=True, normalize_space=True):
    text = unicodedata.normalize("NFC", text)

    if replace_url:
        text = re.sub(r"https?://\S+|www\.\S+", "<URL>", text)

    if replace_email:
        text = re.sub(
            r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b",
            "<EMAIL>",
            text,
        )

    if lowercase:
        text = text.lower()

    if normalize_space:
        text = re.sub(r"\s+", " ", text).strip()

    return text

sample = "  Contact Me: Student@Phenikaa.edu.vn   or visit https://phenikaa-uni.edu.vn  "
print(normalize_text(sample))

contact me: <email> or visit <url>


### Bài tập 6 – Normalization pipeline

Viết `normalize_social_text(text)` thực hiện:
1. Unicode NFC.
2. Lowercase.
3. URL → `<URL>`.
4. Email → `<EMAIL>`.
5. Số nguyên → `<NUM>`.
6. Nhiều khoảng trắng → một khoảng trắng.
7. **Không xóa emoji**.

Test:

```text
"  Tôi mua 2 vé 😄 tại https://abc.vn. Email: Test@abc.vn   "
```

In [56]:
exercise6_text = "  Tôi mua 2 vé 😄 tại https://abc.vn. Email: Test@abc.vn   "

def normalize_social_text(text):
    # 1. Chuẩn hóa Unicode NFC
    text = unicodedata.normalize("NFC", text)
    # 2. Chuẩn hóa URL -> <URL>
    text = re.sub(r"https?://\S+|www\.\S+", "<URL>", text)
    # 3. Chuẩn hóa Email -> <EMAIL>
    text = re.sub(r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b", "<EMAIL>", text)
    # 4. Chuẩn hóa số nguyên -> <NUM>
    text = re.sub(r"\b\d+\b", "<NUM>", text)
    # 5. Chuyển chữ thường (lowercase)
    text = text.lower()
    # 6. Chuẩn hóa khoảng trắng (nhiều khoảng trắng -> 1 khoảng trắng và strip)
    text = re.sub(r"\s+", " ", text).strip()
    # 7. Không xóa emoji (emoji 😄 vẫn được giữ nguyên)
    return text

print("Văn bản gốc  :", repr(exercise6_text))
print("Sau chuẩn hóa:", repr(normalize_social_text(exercise6_text)))


Văn bản gốc  : '  Tôi mua 2 vé 😄 tại https://abc.vn. Email: Test@abc.vn   '
Sau chuẩn hóa: 'tôi mua <num> vé 😄 tại <url> email: <email>'


---
# 8. Minimum Edit Distance

Minimum edit distance là số phép biến đổi tối thiểu để biến chuỗi nguồn thành chuỗi đích.

Ba phép cơ bản:
- insertion,
- deletion,
- substitution.

Với Levenshtein distance, mỗi phép có cost = 1.

In [57]:
pairs = [
    ("kitten", "sitting"),
    ("graffe", "giraffe"),
    ("intention", "execution"),
]

for a, b in pairs:
    print(f"{a:10s} -> {b:10s}: {edit_distance(a, b)}")

kitten     -> sitting   : 3
graffe     -> giraffe   : 1
intention  -> execution : 5


## 8.1. Dynamic Programming

In [58]:
def levenshtein_distance(source, target):
    m, n = len(source), len(target)
    dp = [[0] * (n + 1) for _ in range(m + 1)]

    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j

    for i in range(1, m + 1):
        for j in range(1, n + 1):
            cost = 0 if source[i - 1] == target[j - 1] else 1
            dp[i][j] = min(
                dp[i - 1][j] + 1,       # deletion
                dp[i][j - 1] + 1,       # insertion
                dp[i - 1][j - 1] + cost # substitution/match
            )

    return dp[m][n], dp


distance, matrix = levenshtein_distance("kitten", "sitting")
print("Distance =", distance)

pd.DataFrame(
    matrix,
    index=["∅"] + list("kitten"),
    columns=["∅"] + list("sitting"),
)

Distance = 3


,∅,s,i,t,t,i,n,g
∅,0,1,2,3,4,5,6,7
k,1,1,2,3,4,5,6,7
i,2,2,1,2,3,4,5,6
t,3,3,2,1,2,3,4,5
t,4,4,3,2,1,2,3,4
e,5,5,4,3,2,2,3,4
n,6,6,5,4,3,3,2,3


### Bài tập 7 – Edit distance
Tính cho:
1. `cat` → `cut`
2. `book` → `back`
3. `graffe` → `giraffe`
4. `ngon` → `ngôn`

Sau đó:
- so sánh hàm tự cài đặt với `nltk.edit_distance`;
- giải thích ứng dụng cho spell correction.

In [59]:
exercise7_pairs = [
    ("cat", "cut"),
    ("book", "back"),
    ("graffe", "giraffe"),
    ("ngon", "ngôn"),
]

print("=== SO SÁNH KHOẢNG CÁCH CHỈNH SỬA (EDIT DISTANCE) ===")
print(f"{'Cặp từ (Source -> Target)':<25} | {'DP Custom':<10} | {'NLTK Distance':<15} | {'Trạng thái'}")
print("-" * 65)

for src, tgt in exercise7_pairs:
    dp_dist, _ = levenshtein_distance(src, tgt)
    nltk_dist = edit_distance(src, tgt)
    status = "Khớp 100%" if dp_dist == nltk_dist else "Khác biệt"
    pair_str = f"'{src}' -> '{tgt}'"
    print(f"{pair_str:<25} | {dp_dist:<10} | {nltk_dist:<15} | {status}")

print("\n=== ỨNG DỤNG CHO SPELL CORRECTION (SỬA LỖI CHÍNH TẢ) ===")
print("1. Khi người dùng gõ sai từ (ví dụ 'graffe'), hệ thống tính Edit Distance giữa từ đó với tất cả các từ trong từ điển (Vocabulary).")
print("2. Các từ có khoảng cách nhỏ nhất (thường là distance = 1 hoặc 2, ví dụ 'giraffe' có distance=1) sẽ được ưu tiên xếp hạng hàng đầu để gợi ý hoặc tự động sửa lỗi.")
print("3. Tương tự, với tiếng Việt gõ thiếu dấu ('ngon' -> 'ngôn'), khoảng cách là 1 thao tác thay thế ký tự, giúp gợi ý từ có dấu phù hợp theo ngữ cảnh.")


=== SO SÁNH KHOẢNG CÁCH CHỈNH SỬA (EDIT DISTANCE) ===
Cặp từ (Source -> Target) | DP Custom  | NLTK Distance   | Trạng thái
-----------------------------------------------------------------
'cat' -> 'cut'            | 1          | 1               | Khớp 100%
'book' -> 'back'          | 2          | 2               | Khớp 100%
'graffe' -> 'giraffe'     | 1          | 1               | Khớp 100%
'ngon' -> 'ngôn'          | 1          | 1               | Khớp 100%

=== ỨNG DỤNG CHO SPELL CORRECTION (SỬA LỖI CHÍNH TẢ) ===
1. Khi người dùng gõ sai từ (ví dụ 'graffe'), hệ thống tính Edit Distance giữa từ đó với tất cả các từ trong từ điển (Vocabulary).
2. Các từ có khoảng cách nhỏ nhất (thường là distance = 1 hoặc 2, ví dụ 'giraffe' có distance=1) sẽ được ưu tiên xếp hạng hàng đầu để gợi ý hoặc tự động sửa lỗi.
3. Tương tự, với tiếng Việt gõ thiếu dấu ('ngon' -> 'ngôn'), khoảng cách là 1 thao tác thay thế ký tự, giúp gợi ý từ có dấu phù hợp theo ngữ cảnh.


---
# 9. Preprocessing Pipeline

## 9.1. English

In [60]:
def preprocess_english(text, remove_stopwords=True, lemmatize=True):
    text = unicodedata.normalize("NFC", text)
    text = text.lower()
    text = re.sub(r"https?://\S+", "<URL>", text)
    text = re.sub(r"\s+", " ", text).strip()

    tokens = word_tokenize(text)
    tokens = [t for t in tokens if re.search(r"\w", t)]

    if remove_stopwords:
        sw = set(stopwords.words("english")) - {"not", "no", "nor"}
        tokens = [t for t in tokens if t not in sw]

    if lemmatize:
        lem = WordNetLemmatizer()
        tokens = [lem.lemmatize(t) for t in tokens]

    return tokens

sample_en = "I do not like these movies, but I really love the actors!"
print(preprocess_english(sample_en))

['not', 'like', 'movie', 'really', 'love', 'actor']


## 9.2. Vietnamese

In [61]:
def preprocess_vietnamese(text, remove_stopwords=True):
    text = unicodedata.normalize("NFC", text)
    text = text.lower()
    text = re.sub(r"https?://\S+", "<URL>", text)
    text = re.sub(r"\s+", " ", text).strip()

    tokens = vi_word_tokenize(text)

    if remove_stopwords:
        vi_stop = {"là", "và", "của", "ở", "có", "một", "những", "các", "cho", "với"}
        tokens = [t for t in tokens if t not in vi_stop]

    return tokens

sample_vi = "Xử lý ngôn ngữ tự nhiên là một lĩnh vực quan trọng của trí tuệ nhân tạo."
print(preprocess_vietnamese(sample_vi))

['xử lý', 'ngôn ngữ', 'tự nhiên', 'lĩnh vực', 'quan trọng', 'trí tuệ', 'nhân tạo', '.']


---
# 10. Mini-lab – Vocabulary trước và sau preprocessing

In [62]:
docs = [
    "I love natural language processing.",
    "Natural language processing is interesting.",
    "I do not like poor tokenization.",
    "Tokenization and normalization are important for NLP.",
]

for i, doc in enumerate(docs, 1):
    print(i, preprocess_english(doc))

1 ['love', 'natural', 'language', 'processing']
2 ['natural', 'language', 'processing', 'interesting']
3 ['not', 'like', 'poor', 'tokenization']
4 ['tokenization', 'normalization', 'important', 'nlp']


### Bài tập 8
1. Tạo vocabulary từ dữ liệu thô.
2. Tạo vocabulary sau preprocessing.
3. So sánh kích thước vocabulary.
4. In 10 token phổ biến nhất.
5. Nhận xét thông tin nào bị mất và thông tin nào được chuẩn hóa.

In [63]:
# 1. Tạo vocabulary từ dữ liệu thô (tách theo khoảng trắng đơn giản, không chuẩn hóa)
raw_tokens = []
for d in docs:
    raw_tokens.extend(d.split())
raw_vocab = set(raw_tokens)
raw_counts = Counter(raw_tokens)

# 2. Tạo vocabulary sau preprocessing (dùng pipeline preprocess_english đã định nghĩa)
clean_tokens = []
for d in docs:
    clean_tokens.extend(preprocess_english(d))
clean_vocab = set(clean_tokens)
clean_counts = Counter(clean_tokens)

# 3. So sánh kích thước vocabulary
print("=== SO SÁNH KÍCH THƯỚC VOCABULARY ===")
print(f"- Số token thô (raw tokens)       : {len(raw_tokens)} | Số từ vựng duy nhất (raw vocab)  : {len(raw_vocab)}")
print(f"- Số token sạch (clean tokens)    : {len(clean_tokens)} | Số từ vựng duy nhất (clean vocab): {len(clean_vocab)}")
print(f"- Giảm được: {len(raw_vocab) - len(clean_vocab)} từ vựng ({((len(raw_vocab) - len(clean_vocab)) / len(raw_vocab)):.1%})")

# 4. In các token phổ biến nhất
print("\nTop token phổ biến nhất trong Clean Data:")
for tok, count in clean_counts.most_common(10):
    print(f"  + '{tok}': {count} lần")

# 5. Nhận xét
print("\n=== NHẬN XÉT ===")
print("1. Thông tin được chuẩn hóa:")
print("   - Chuyển về chữ thường (lowercase): gom 'Natural' và 'natural' thành 1 từ duy nhất.")
print("   - Lemmatization & xóa dấu câu: loại bỏ các dấu chấm '.' dính vào đuôi từ ('processing.' -> 'processing').")
print("2. Thông tin bị loại bỏ:")
print("   - Các stopwords phổ biến như 'is', 'are', 'and', 'for' bị loại bỏ nhằm giảm chiều dữ liệu và tập trung vào các từ mang nội dung chính.")
print("   - Giữ lại từ phủ định 'not' để bảo toàn ngữ nghĩa.")


=== SO SÁNH KÍCH THƯỚC VOCABULARY ===
- Số token thô (raw tokens)       : 23 | Số từ vựng duy nhất (raw vocab)  : 21
- Số token sạch (clean tokens)    : 16 | Số từ vựng duy nhất (clean vocab): 12
- Giảm được: 9 từ vựng (42.9%)

Top token phổ biến nhất trong Clean Data:
  + 'natural': 2 lần
  + 'language': 2 lần
  + 'processing': 2 lần
  + 'tokenization': 2 lần
  + 'love': 1 lần
  + 'interesting': 1 lần
  + 'not': 1 lần
  + 'like': 1 lần
  + 'poor': 1 lần
  + 'normalization': 1 lần

=== NHẬN XÉT ===
1. Thông tin được chuẩn hóa:
   - Chuyển về chữ thường (lowercase): gom 'Natural' và 'natural' thành 1 từ duy nhất.
   - Lemmatization & xóa dấu câu: loại bỏ các dấu chấm '.' dính vào đuôi từ ('processing.' -> 'processing').
2. Thông tin bị loại bỏ:
   - Các stopwords phổ biến như 'is', 'are', 'and', 'for' bị loại bỏ nhằm giảm chiều dữ liệu và tập trung vào các từ mang nội dung chính.
   - Giữ lại từ phủ định 'not' để bảo toàn ngữ nghĩa.


---
# 11. Bài tập tổng hợp – Corpus song ngữ

In [64]:
mixed_corpus = [
    ("en", "NLP is changing the way we build software!"),
    ("en", "I don't like noisy text... Visit https://example.com."),
    ("vi", "Xử lý ngôn ngữ tự nhiên đang thay đổi cách chúng ta xây dựng phần mềm."),
    ("vi", "Tôi không thích dữ liệu văn bản quá nhiễu 😄."),
]

### Bài tập 9
Viết:

```python
preprocess(text, lang)
```

Yêu cầu:
- `lang="en"` → English tokenizer.
- `lang="vi"` → Vietnamese word segmentation.
- Unicode NFC.
- Lowercase.
- URL normalization.
- Whitespace normalization.
- Loại punctuation độc lập.
- Stopword removal là tùy chọn.
- Trả về danh sách token.

Sau đó tạo DataFrame:

| lang | original | normalized_tokens | n_tokens |
|---|---|---|---|

In [65]:
def preprocess(text, lang="en", remove_stopwords=True):
    # Chuẩn hóa Unicode NFC
    text = unicodedata.normalize("NFC", text)
    # URL normalization
    text = re.sub(r"https?://\S+|www\.\S+", "<URL>", text)
    # Lowercase
    text = text.lower()
    # Whitespace normalization
    text = re.sub(r"\s+", " ", text).strip()
    
    if lang == "en":
        # English Tokenization
        tokens = word_tokenize(text)
        # Loại punctuation độc lập (giữ token có chứa chữ/số hoặc thẻ <URL>)
        tokens = [t for t in tokens if re.search(r"\w|<url>", t)]
        if remove_stopwords:
            sw = set(stopwords.words("english")) - {"not", "no", "nor"}
            tokens = [t for t in tokens if t not in sw]
            lem = WordNetLemmatizer()
            tokens = [lem.lemmatize(t) for t in tokens]
    elif lang == "vi":
        # Vietnamese Word Segmentation
        tokens = vi_word_tokenize(text)
        # Loại punctuation độc lập (giữ token có chữ hoặc emoji)
        tokens = [t for t in tokens if re.search(r"\w|<url>", t) or any(c in t for c in "😄😃😊😀")]
        if remove_stopwords:
            vi_stop = {"là", "và", "của", "ở", "có", "một", "những", "các", "cho", "với", "quá", "đang"}
            tokens = [t for t in tokens if t not in vi_stop]
    else:
        tokens = text.split()
        
    return tokens

# Tạo DataFrame tổng hợp từ mixed_corpus
records = []
for lang, original in mixed_corpus:
    norm_tokens = preprocess(original, lang=lang, remove_stopwords=True)
    records.append({
        "lang": lang,
        "original": original,
        "normalized_tokens": norm_tokens,
        "n_tokens": len(norm_tokens)
    })

df_corpus = pd.DataFrame(records)
print("=== BẢNG TỔNG HỢP PREPROCESSING CORPUS SONG NGỮ ===")
display(df_corpus)


=== BẢNG TỔNG HỢP PREPROCESSING CORPUS SONG NGỮ ===


,lang,original,normalized_tokens,n_tokens
0,en,NLP is changing the way we build software!,"[nlp, changing, way, build, software]",5
1,en,I don't like noisy text... Visit https://examp...,"[n't, like, noisy, text, visit, url]",6
2,vi,Xử lý ngôn ngữ tự nhiên đang thay đổi cách chú...,"[xử lý, ngôn ngữ, tự nhiên, thay đổi, cách, ch...",8
3,vi,Tôi không thích dữ liệu văn bản quá nhiễu 😄.,"[tôi, không, thích, dữ liệu, văn bản, quá nhiễ...",7


### Bài tập 10 – Phân tích lỗi tokenization
Tìm **5 ví dụ thực tế** tokenizer đơn giản có thể xử lý sai, thuộc ít nhất 3 nhóm:
- abbreviation,
- URL/email,
- emoji,
- hashtag,
- số thập phân,
- ngày tháng,
- hyphenated words,
- từ đa âm tiết tiếng Việt,
- tên riêng.

Với mỗi ví dụ:
1. Input.
2. Tokenization mong muốn.
3. Tokenization thực tế.
4. Lỗi.
5. Cách cải tiến.

---
# 12. Challenge – Spell Correction bằng Edit Distance

In [66]:
vocab = [
    "language", "processing", "tokenization", "normalization",
    "distance", "natural", "computer", "learning",
    "giraffe", "student"
]

def suggest(word, vocab, k=3):
    # Tính edit distance từ từ cần sửa (word) tới từng từ trong vocab
    ranked = sorted(
        ((candidate, edit_distance(word.lower(), candidate.lower())) for candidate in vocab),
        key=lambda item: (item[1], item[0])  # ưu tiên khoảng cách nhỏ nhất, sau đó theo alphabet
    )
    return ranked[:k]

# Chạy thử nghiệm các trường hợp gõ sai:
test_words = ["langauge", "processin", "toknization", "graffe", "studnt"]
print("=== KẾT QUẢ GỢI Ý CHÍNH TẢ (SPELL CORRECTION SUGGESTIONS) ===")
for tw in test_words:
    suggestions = suggest(tw, vocab, k=3)
    sug_str = ", ".join([f"'{c}' (d={d})" for c, d in suggestions])
    print(f"Input: '{tw:<12}' -> Gợi ý: {sug_str}")


=== KẾT QUẢ GỢI Ý CHÍNH TẢ (SPELL CORRECTION SUGGESTIONS) ===
Input: 'langauge    ' -> Gợi ý: 'language' (d=2), 'distance' (d=6), 'giraffe' (d=6)
Input: 'processin   ' -> Gợi ý: 'processing' (d=1), 'computer' (d=8), 'learning' (d=8)
Input: 'toknization ' -> Gợi ý: 'tokenization' (d=1), 'normalization' (d=5), 'computer' (d=9)
Input: 'graffe      ' -> Gợi ý: 'giraffe' (d=1), 'distance' (d=6), 'language' (d=6)
Input: 'studnt      ' -> Gợi ý: 'student' (d=1), 'natural' (d=5), 'distance' (d=6)


---
# 13. Câu hỏi cuối buổi
1. Sentence segmentation và word tokenization khác nhau thế nào?
2. Vì sao word segmentation tiếng Việt không thể chỉ dựa vào whitespace?
3. Khi nào không nên remove stopwords?
4. Stemming khác lemmatization ở đâu?
5. Có nên lowercase mọi văn bản không?
6. Edit distance đo cái gì?
7. Vì sao preprocessing phụ thuộc downstream task?
8. Với BERT/LLM, tokenization có còn quan trọng không?

---
# 14. Đáp án tham khảo cho giảng viên
> Có thể xóa phần này trước khi giao notebook cho sinh viên.

## Gợi ý Bài tập 4
Với sentiment analysis, nên cân nhắc giữ `not`, `no`, `nor`.

In [67]:
text = exercise4_text
tokens = [t.lower() for t in word_tokenize(text) if t.isalpha()]

default_sw = set(stopwords.words("english"))
keep_negation_sw = default_sw - {"not", "no", "nor"}

print("Default:", [t for t in tokens if t not in default_sw])
print("Keep negation:", [t for t in tokens if t not in keep_negation_sw])

Default: ['movie', 'good', 'actors', 'good']
Keep negation: ['movie', 'not', 'good', 'actors', 'good']


## Gợi ý Bài tập 6

In [68]:
def normalize_social_text_solution(text):
    text = unicodedata.normalize("NFC", text)
    text = re.sub(r"https?://\S+|www\.\S+", "<URL>", text)
    text = re.sub(
        r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b",
        "<EMAIL>",
        text,
    )
    text = re.sub(r"\b\d+\b", "<NUM>", text)
    text = text.lower()
    text = re.sub(r"\s+", " ", text).strip()
    return text

print(normalize_social_text_solution(exercise6_text))

tôi mua <num> vé 😄 tại <url> email: <email>


## Gợi ý Bài tập 7

In [69]:
for a, b in exercise7_pairs:
    d1, _ = levenshtein_distance(a, b)
    d2 = edit_distance(a, b)
    print(f"{a:8s} -> {b:8s} | DP={d1} | NLTK={d2}")

cat      -> cut      | DP=1 | NLTK=1
book     -> back     | DP=2 | NLTK=2
graffe   -> giraffe  | DP=1 | NLTK=1
ngon     -> ngôn     | DP=1 | NLTK=1


## Gợi ý Challenge

In [70]:
def suggest_solution(word, vocab, k=3):
    ranked = sorted(
        ((candidate, edit_distance(word, candidate)) for candidate in vocab),
        key=lambda x: (x[1], x[0])
    )
    return ranked[:k]

print(suggest_solution("langauge", vocab, 3))

[('language', 2), ('distance', 6), ('giraffe', 6)]


---
# Tài liệu tham khảo
- Daniel Jurafsky & James H. Martin, *Speech and Language Processing*, Chapter 2.
- NLTK documentation.
- underthesea documentation.
- Tài liệu bài giảng **Text Processing basic** của học phần NLP.